# Revision B — re-evaluating a published defence, and NTU RGB+D 120

Addresses reviewer concerns **M2** (no published defence is re-evaluated) and
**M5** (single dataset).

Section 1 is the single highest-value experiment remaining. It reimplements the
anonymiser of Moon et al. (AAAI 2023) from their published specification, then
evaluates it twice: once the way they did, with classifiers pre-trained on
original skeletons and never retrained, and once with our attacker ladder.

Their reported figure is 4.20% re-identification after anonymisation. Our thesis
predicts that figure will not survive a retrained attacker. If it does survive,
we report that honestly and the paper becomes a validation study.

| Section | Reviewer point | Trains? | Time |
|---|---|---|---|
| 1 | **M2** reimplement their anonymiser | yes | ~2 h |
| 2 | **M2** evaluate it their way vs our way | yes | ~1.5 h |
| 3 | **M5** NTU-120 download and cache | no | ~40 min |
| 4 | **M5** NTU-120 core experiments | yes | ~5 h |
| 5 | Assemble tables | no | seconds |

## 0 · Setup

## 0 · Setup — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0 · Setup — paths and constants

In [ ]:
import os, sys, re, json, time, math, pickle, random
import numpy as np

SEED = 1
random.seed(SEED); np.random.seed(SEED)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/PhD_Q1'
    SCRATCH = '/content/scratch'
elif IN_KAGGLE:
    PROJECT = '/kaggle/working/PhD_Q1'
    SCRATCH = '/kaggle/working/scratch'
else:
    PROJECT = os.path.abspath('./PhD_Q1')
    SCRATCH = os.path.abspath('./scratch')

DATA_DIR = os.path.join(PROJECT, 'data')
CKPT_DIR = os.path.join(PROJECT, 'checkpoints')
RES_DIR  = os.path.join(PROJECT, 'results')
for d in (DATA_DIR, CKPT_DIR, RES_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)

PKL_PATH  = os.path.join(DATA_DIR, 'ntu60_3danno.pkl')
CACHE_X   = os.path.join(DATA_DIR, 'ntu60_T64_x.npy')
CACHE_M   = os.path.join(DATA_DIR, 'ntu60_T64_meta.npz')

# geometry / tensor shapes
NUM_NODE, MAX_PERSON, T_LEN, IN_CH = 25, 2, 64, 3

print('project :', PROJECT)
print('pkl     :', PKL_PATH, '->', os.path.exists(PKL_PATH))

## 0 · Setup — torch and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime > Change runtime type > GPU.')

## 1 · Load the NTU annotations

In [ ]:
t0 = time.time()
with open(PKL_PATH, 'rb') as f:
    D = pickle.load(f)
print('loaded in %.1f s' % (time.time() - t0))

anns  = D['annotations']
names = [a['frame_dir'] for a in anns]

setup   = np.array([int(n[1:4])   for n in names], dtype=np.int16)
camera  = np.array([int(n[5:8])   for n in names], dtype=np.int16)
subject = np.array([int(n[9:12])  for n in names], dtype=np.int16)
action  = np.array([a['label']    for a in anns],  dtype=np.int16)

print('samples  :', len(anns))
print('subjects :', len(np.unique(subject)))
print('cameras  :', np.unique(camera))
print('actions  :', len(np.unique(action)))
print('example  :', names[0])
print('keys     :', list(anns[0].keys()))
print('keypoint :', np.asarray(anns[0]['keypoint']).shape, ' (M, T, V, C)')

assert len(np.unique(subject)) == 40, 'expected 40 subjects'
print('\nOK — identity labels present. Proceed.')

## 2 · Preprocessing functions

In [ ]:
def _unit(v):
    n = np.linalg.norm(v)
    return v if n < 1e-8 else v / n

def _rotation_matrix(axis, theta):
    axis = _unit(np.asarray(axis, dtype=np.float64))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([[aa+bb-cc-dd, 2*(bc+ad),   2*(bd-ac)],
                     [2*(bc-ad),   aa+cc-bb-dd, 2*(cd+ab)],
                     [2*(bd+ac),   2*(cd-ab),   aa+dd-bb-cc]], dtype=np.float32)

def _angle_between(v1, v2):
    return np.arccos(np.clip(np.dot(_unit(v1), _unit(v2)), -1.0, 1.0))

def _align(skel, ja, jb, target):
    # rotate the whole sequence so that joint ja -> jb points along `target`
    vec = skel[0, 0, jb] - skel[0, 0, ja]
    if np.linalg.norm(vec) < 1e-8:
        return skel
    axis, angle = np.cross(vec, target), _angle_between(vec, target)
    if np.linalg.norm(axis) < 1e-8 or angle < 1e-8:
        return skel
    return (skel @ _rotation_matrix(axis, angle).T).astype(np.float32)

def pre_normalize(kp):
    kp = np.asarray(kp, dtype=np.float32).copy()
    valid = kp.sum(axis=(0, 2, 3)) != 0
    if valid.any():
        kp = kp[:, valid]
    if kp.shape[1] == 0:
        return np.zeros((1, 1, NUM_NODE, 3), dtype=np.float32)
    mask = kp.sum(axis=-1) != 0
    kp[mask] = kp[mask] - kp[0, 0, 1].copy()          # centre on spine
    kp = _align(kp, 0, 1, np.array([0, 0, 1], np.float32))   # spine     -> z
    kp = _align(kp, 8, 4, np.array([1, 0, 0], np.float32))   # shoulders -> x
    return kp

def resample(kp, t_len=T_LEN):
    M, T, V, Ch = kp.shape
    out = kp[:, np.linspace(0, T - 1, t_len).round().astype(np.int64)]
    if M < MAX_PERSON:
        out = np.concatenate(
            [out, np.zeros((MAX_PERSON - M, t_len, V, Ch), out.dtype)], axis=0)
    return out[:MAX_PERSON]

# quick self-test on synthetic data
_t = np.random.RandomState(0).normal(size=(1, 90, 25, 3)).astype(np.float32)
assert resample(pre_normalize(_t)).shape == (MAX_PERSON, T_LEN, NUM_NODE, 3)
print('preprocessing OK')

## 2 · Build or load the cache

In [ ]:
if os.path.exists(CACHE_X) and os.path.exists(CACHE_M):
    print('cache found, skipping preprocessing')
else:
    N = len(anns)
    X = np.zeros((N, MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    t0 = time.time()
    for i, a in enumerate(anns):
        X[i] = resample(pre_normalize(np.asarray(a['keypoint']))).astype(np.float16)
        if (i + 1) % 5000 == 0:
            el = time.time() - t0
            print('  %6d / %d   %.0f s elapsed, ~%.0f s left'
                  % (i + 1, N, el, el / (i + 1) * (N - i - 1)))
    np.save(CACHE_X, X)
    np.savez(CACHE_M, setup=setup, camera=camera, subject=subject,
             action=action, names=np.array(names))
    print('cached in %.0f s -> %.2f GB' % (time.time() - t0, X.nbytes / 1e9))
    del X

## 2 · Copy the cache to local disk

In [ ]:
# copy to local disk first: Drive I/O is slow for repeated reads
LOCAL_X = os.path.join(SCRATCH, 'x.npy')
if not os.path.exists(LOCAL_X):
    import shutil; shutil.copy(CACHE_X, LOCAL_X)

X = np.load(LOCAL_X)                       # (N, 2, 64, 25, 3) float16
_m = np.load(CACHE_M, allow_pickle=True)
setup, camera, subject, action = _m['setup'], _m['camera'], _m['subject'], _m['action']
names = list(_m['names'])

SUBJ_IDS   = sorted(np.unique(subject).tolist())
SUBJ_LABEL = np.array([SUBJ_IDS.index(s) for s in subject], dtype=np.int64)  # 0..39
ACT_LABEL  = action.astype(np.int64)                                         # 0..59

print('X', X.shape, X.dtype, '%.2f GB' % (X.nbytes / 1e9))
print('action classes', ACT_LABEL.max() + 1, '| identity classes', SUBJ_LABEL.max() + 1)

## 3 · Evaluation protocols (splits)

In [ ]:
XSUB_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38]

SPLITS = {}
SPLITS['xsub'] = (np.where(np.isin(subject, XSUB_TRAIN))[0],
                  np.where(~np.isin(subject, XSUB_TRAIN))[0])

rs, tr, te = np.random.RandomState(SEED), [], []
for s in SUBJ_IDS:                                   # stratified 80/20 per person
    idx = np.where(subject == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr += idx[:k].tolist(); te += idx[k:].tolist()
SPLITS['reid_rand'] = (np.array(sorted(tr)), np.array(sorted(te)))

SPLITS['reid_xview'] = (np.where(np.isin(camera, [1, 2]))[0],
                        np.where(camera == 3)[0])

for k, (a, b) in SPLITS.items():
    print('%-11s train %6d  test %6d  | subjects %2d / %2d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b]))))

assert len(set(subject[SPLITS['xsub'][0]]) & set(subject[SPLITS['xsub'][1]])) == 0
assert len(set(subject[SPLITS['reid_xview'][0]]) &
           set(subject[SPLITS['reid_xview'][1]])) == 40

## 4 · ST-GCN definition

In [ ]:
_INWARD_1B = [(1,2),(2,21),(3,21),(4,3),(5,21),(6,5),(7,6),(8,7),(9,21),
              (10,9),(11,10),(12,11),(13,1),(14,13),(15,14),(16,15),
              (17,1),(18,17),(19,18),(20,19),(22,23),(23,8),(24,25),(25,12)]
INWARD    = [(i-1, j-1) for i, j in _INWARD_1B]
OUTWARD   = [(j, i) for i, j in INWARD]
SELF_LINK = [(i, i) for i in range(NUM_NODE)]

def _edge2mat(link, n):
    A = np.zeros((n, n), dtype=np.float32)
    for i, j in link:
        A[j, i] = 1.0
    return A

def _norm_digraph(A):
    Dl, Dn = A.sum(0), np.zeros_like(A)
    for i in range(A.shape[0]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** -1.0
    return A @ Dn

def build_adjacency():
    return np.stack([_edge2mat(SELF_LINK, NUM_NODE),
                     _norm_digraph(_edge2mat(INWARD, NUM_NODE)),
                     _norm_digraph(_edge2mat(OUTWARD, NUM_NODE))]).astype(np.float32)


class SpatialGC(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.K = A.shape[0]
        self.conv = nn.Conv2d(in_c, out_c * self.K, 1)
        self.register_buffer('A', torch.from_numpy(A.copy()))

    def forward(self, x):
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('nkctv,kvw->nctw', x, self.A.to(x.dtype)).contiguous()


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, residual=True):
        super().__init__()
        self.gcn = SpatialGC(in_c, out_c, A)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(0.1)
        if not residual:
            self.res = None
        elif in_c == out_c and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)),
                                     nn.BatchNorm2d(out_c))

    def forward(self, x):
        r = 0 if self.res is None else self.res(x)
        y = F.relu(self.bn1(self.gcn(x)))
        y = self.drop(self.bn2(self.tcn(y)))
        return F.relu(y + r)


class STGCN(nn.Module):
    def __init__(self, num_class, in_channels=IN_CH, num_person=MAX_PERSON, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * NUM_NODE)
        cfg = [(in_channels, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True),
               (64, 64, 1, True), (64, 128, 2, True), (128, 128, 1, True),
               (128, 128, 1, True), (128, 256, 2, True), (256, 256, 1, True),
               (256, 256, 1, True)]
        self.layers = nn.ModuleList([STGCNBlock(i, o, A, s, r) for i, o, s, r in cfg])
        self.drop, self.fc, self.feat_dim = nn.Dropout(dropout), nn.Linear(256, num_class), 256

    def extract(self, x):                      # x: N, C, T, V, M
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        for l in self.layers:
            x = l(x)
        return F.adaptive_avg_pool2d(x, 1).view(N, M, -1).mean(1)

    def forward(self, x, return_feat=False):
        f = self.extract(x)
        out = self.fc(self.drop(f))
        return (out, f) if return_feat else out


_m = STGCN(60)
_o, _f = _m(torch.randn(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON), return_feat=True)
print('logits', tuple(_o.shape), '| features', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o, _f

## 5 · Training engine

In [ ]:
def _euler(theta):
    cx, cy, cz = np.cos(theta); sx, sy, sz = np.sin(theta)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def augment(x, rot=0.3, scale=0.1):
    R = _euler(np.random.uniform(-rot, rot, 3))
    s = 1.0 + np.random.uniform(-scale, scale)
    mask = (x.sum(-1) != 0)[..., None]
    return np.where(mask, (x @ R.T) * s, 0.0).astype(np.float32)


class SkeletonDS(torch.utils.data.Dataset):
    def __init__(self, X, y, idx, train=False):
        self.X, self.y, self.idx, self.train = X, y, idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j].astype(np.float32)
        if self.train:
            x = augment(x)
        x = np.ascontiguousarray(np.transpose(x, (3, 1, 2, 0)))   # C,T,V,M
        return torch.from_numpy(x), int(self.y[j])


def _load_ckpt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(p, map_location='cpu')


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.numel()
    return 100.0 * correct / total


def run_experiment(tag, labels, split, num_classes, epochs=60, lr=0.1, bs=64,
                   wd=5e-4, warmup=5, workers=2, init_from=None, freeze=False):
    tr_idx, te_idx = SPLITS[split]
    tl = torch.utils.data.DataLoader(SkeletonDS(X, labels, tr_idx, True), bs, shuffle=True,
                                     num_workers=workers, pin_memory=(DEVICE == 'cuda'),
                                     drop_last=True, persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(SkeletonDS(X, labels, te_idx, False), bs * 2,
                                     shuffle=False, num_workers=workers,
                                     pin_memory=(DEVICE == 'cuda'))

    model = STGCN(num_classes).to(DEVICE)
    if init_from:
        sd = _load_ckpt(init_from)['model']
        sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
        print('  init from', os.path.basename(init_from),
              model.load_state_dict(sd, strict=False).missing_keys[:2], '...')
    if freeze:
        for n_, p_ in model.named_parameters():
            if not n_.startswith('fc.'):
                p_.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] split=%s  classes=%d  train=%d  test=%d'
          % (tag, split, num_classes, len(tr_idx), len(te_idx)))
    for ep in range(start, epochs):
        cur_lr = set_lr(ep); model.train(); t0 = time.time(); run = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        acc = evaluate(model, vl)
        hist.append({'epoch': ep, 'loss': run / max(1, len(tl)), 'acc': acc, 'lr': cur_lr})
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'best': max(best, acc), 'hist': hist}, last)
        if acc > best:
            best = acc; torch.save({'model': model.state_dict(), 'acc': acc}, best_p)
        print('  ep %3d/%d  loss %.3f  acc %6.2f%%  best %6.2f%%  lr %.4f  (%.0fs)'
              % (ep + 1, epochs, hist[-1]['loss'], acc, best, cur_lr, time.time() - t0))

    json.dump(hist, open(os.path.join(RES_DIR, tag + '_history.json'), 'w'), indent=1)
    print('[%s] BEST %.2f%%' % (tag, best))
    return {'tag': tag, 'split': split, 'classes': num_classes,
            'best_acc': best, 'ckpt': best_p}

RESULTS = {}
RES_JSON = os.path.join(RES_DIR, 'phase1_results.json')
if os.path.exists(RES_JSON):
    RESULTS = json.load(open(RES_JSON))
    print('loaded previous results:', list(RESULTS))

def save_results():
    json.dump(RESULTS, open(RES_JSON, 'w'), indent=2)

In [ ]:
import json, os, glob, math, time
RES_B = os.path.join(RES_DIR, 'revisionB')
os.makedirs(RES_B, exist_ok=True)
RB_PATH = os.path.join(RES_B, 'revisionB.json')
RB = json.load(open(RB_PATH)) if os.path.exists(RB_PATH) else {}
def save_rb():
    json.dump(RB, open(RB_PATH, 'w'), indent=2)

RESULTS = json.load(open(os.path.join(RES_DIR, 'phase1_results.json')))

def find_ckpt(tag):
    for r in [CKPT_DIR, '/content/drive/MyDrive/PhD_Q1/checkpoints']:
        if not os.path.isdir(r):
            continue
        for suf in ['_best.pt', '_last.pt']:
            p = os.path.join(r, tag + suf)
            if os.path.exists(p):
                return p
    h = glob.glob('/content/drive/MyDrive/**/%s_best.pt' % tag, recursive=True)
    return h[0] if h else None

for key in list(RESULTS):
    p = find_ckpt(RESULTS[key].get('tag', key))
    if p:
        RESULTS[key]['ckpt'] = p

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
print('ready')

## 1 · M2 — reimplement the anonymiser of Moon et al.

Their specification, from the published paper:

- Anonymiser `f(x) = MLP(x) + x`, applied per frame, two fully-connected layers,
  weights initialised near zero so it begins as a small perturbation
  (their "residual" variant).
- Objective: minimise `CE(y, h(f(x))) - alpha * H(g(f(x))) + beta * ||x - f(x)||^2`
  over the anonymiser, where `H` is the entropy of the privacy classifier's
  output. They use entropy rather than negative cross-entropy specifically to
  avoid confidently-wrong predictions from which the label could be inverted.
- Maximisation step: update the privacy classifier `g` by its own cross-entropy.
- The action classifier `h` is pre-trained and **frozen**.
- For NTU-60 with the residual anonymiser they report alpha = 1, beta = 10.

We follow this exactly, substituting our ST-GCN for their Shift-GCN, and our
`reid_rand` identity split for their camera split. Both substitutions are stated
in the manuscript; neither affects the evaluation question.

In [ ]:
class ResidualAnonymizer(nn.Module):
    """f(x) = MLP(x) + x, applied independently to each frame."""
    def __init__(self, n_joint=NUM_NODE, c=IN_CH, hidden=128):
        super().__init__()
        d = n_joint * c
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(inplace=True),
                                 nn.Linear(hidden, d))
        for m in self.net:                      # start as a near-identity map
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=1e-3); nn.init.zeros_(m.bias)
        self.V, self.C = n_joint, c

    def forward(self, x):                       # x: N, C, T, V, M
        N, Cc, T, V, M = x.shape
        z = x.permute(0, 4, 2, 3, 1).reshape(N * M * T, V * Cc)
        z = self.net(z).reshape(N, M, T, V, Cc).permute(0, 4, 2, 3, 1)
        return x + z


def train_anonymizer(alpha=1.0, beta=10.0, epochs=40, lr=1e-3, bs=64,
                     workers=4, tag='MOON_anon'):
    tr_idx, te_idx = SPLITS['reid_rand']
    tl = torch.utils.data.DataLoader(
        DualDS(X, ACT_LABEL, SUBJ_LABEL, np.sort(tr_idx), True), bs, shuffle=True,
        num_workers=workers, pin_memory=(DEVICE == 'cuda'), drop_last=True)

    # frozen, pre-trained action classifier (their h)
    h = BASE_STGCN(60).to(DEVICE) if 'BASE_STGCN' in globals() else STGCN(60).to(DEVICE)
    sd = _load_ckpt(RESULTS['A_action_xsub']['ckpt'])
    h.load_state_dict(sd.get('model', sd), strict=False)
    for p in h.parameters():
        p.requires_grad = False
    h.eval()

    # privacy classifier (their g), trained alongside
    g = (BASE_STGCN(40) if 'BASE_STGCN' in globals() else STGCN(40)).to(DEVICE)
    f = ResidualAnonymizer().to(DEVICE)

    opt_f = torch.optim.Adam(f.parameters(), lr=lr)
    opt_g = torch.optim.Adam(g.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    start, hist = 0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        f.load_state_dict(ck['f']); g.load_state_dict(ck['g'])
        opt_f.load_state_dict(ck['opt_f']); opt_g.load_state_dict(ck['opt_g'])
        start, hist = ck['epoch'] + 1, ck['hist']
        print('  resumed at epoch %d' % start)

    print('[%s] alpha=%g beta=%g  train=%d' % (tag, alpha, beta, len(tr_idx)))
    for ep in range(start, epochs):
        f.train(); g.train(); t0 = time.time()
        la = lh = lr_ = 0.0
        for xb, ya, yi in tl:
            xb, ya, yi = xb.to(DEVICE), ya.to(DEVICE), yi.to(DEVICE)

            # ---- minimisation over the anonymiser (their Eq. 2) ----
            opt_f.zero_grad(set_to_none=True)
            xa = f(xb)
            logits_a = h(xa)
            logits_p = g(xa)
            logp = torch.log_softmax(logits_p, dim=1)
            H = -(logp.exp() * logp).sum(1).mean()
            rec = ((xb - xa) ** 2).mean()
            loss_f = crit(logits_a, ya) - alpha * H + beta * rec
            loss_f.backward(); opt_f.step()

            # ---- maximisation step: update the privacy classifier ----
            opt_g.zero_grad(set_to_none=True)
            loss_g = crit(g(f(xb).detach()), yi)
            loss_g.backward(); opt_g.step()

            la += crit(logits_a, ya).item(); lh += H.item(); lr_ += rec.item()

        hist.append({'epoch': ep, 'ce_action': la / len(tl),
                     'entropy': lh / len(tl), 'recon': lr_ / len(tl)})
        torch.save({'f': f.state_dict(), 'g': g.state_dict(),
                    'opt_f': opt_f.state_dict(), 'opt_g': opt_g.state_dict(),
                    'epoch': ep, 'hist': hist}, last)
        print('  ep %2d/%d  CE(action) %.3f  H(adv) %.3f  recon %.5f  (%ds)'
              % (ep + 1, epochs, la / len(tl), lh / len(tl), lr_ / len(tl),
                 time.time() - t0))

    torch.save({'f': f.state_dict(), 'hist': hist},
               os.path.join(CKPT_DIR, tag + '_final.pt'))
    return {'tag': tag, 'alpha': alpha, 'beta': beta,
            'ckpt': os.path.join(CKPT_DIR, tag + '_final.pt'), 'hist': hist}


if 'BASE_STGCN' not in globals():
    BASE_STGCN = STGCN

if 'anonymizer' not in RB:
    RB['anonymizer'] = train_anonymizer()
    save_rb()
print('\nanonymiser trained:', RB['anonymizer']['ckpt'])

## 2 · M2 — their evaluation versus ours

**Their protocol.** A privacy classifier pre-trained on *original* skeletons is
applied to anonymised ones; it is never retrained. This is what produces the
4.20% figure in their Table 1.

**Our protocol.** The attacker is given the anonymised data and retrained on it:
a convex linear probe and an MLP probe on frozen features of a re-trained
encoder, and a from-scratch re-identification network.

The gap between the two numbers is the quantity this paper is about.

In [ ]:
@torch.no_grad()
def anonymise_array(anon_ckpt, chunk=512):
    """Apply the trained anonymiser to the whole cached array."""
    f = ResidualAnonymizer().to(DEVICE)
    f.load_state_dict(_load_ckpt(anon_ckpt)['f']); f.eval()
    out = np.zeros(X.shape, dtype=np.float16)
    ds = SkeletonDS(X, SUBJ_LABEL, np.arange(len(X)), False)
    ld = torch.utils.data.DataLoader(ds, chunk, shuffle=False, num_workers=4)
    i = 0
    for xb, _ in ld:
        xa = f(xb.to(DEVICE)).cpu().numpy()
        # back to the cached layout (N, M, T, V, C)
        xa = np.transpose(xa, (0, 4, 2, 3, 1))
        out[i:i + len(xa)] = xa.astype(np.float16)
        i += len(xa)
        if i % (chunk * 20) == 0:
            print('  %d / %d' % (i, len(X)), end='\r')
    del f
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return out


ANON_PATH = '/content/ntu60_T64_anon.npy'
if os.path.exists(ANON_PATH):
    X_ANON = np.load(ANON_PATH, mmap_mode='r')
    print('loaded cached anonymised array', X_ANON.shape)
else:
    np.save(ANON_PATH, anonymise_array(RB['anonymizer']['ckpt']))
    X_ANON = np.load(ANON_PATH, mmap_mode='r')
    print('\nwritten', ANON_PATH)

# ---- THEIR evaluation: fixed classifier, never retrained ----
@torch.no_grad()
def fixed_classifier_eval(ckpt, Xeval, idx, labels, num_classes=40, bs=256):
    m = BASE_STGCN(num_classes).to(DEVICE)
    sd = _load_ckpt(ckpt)
    m.load_state_dict(sd.get('model', sd), strict=False); m.eval()
    ld = torch.utils.data.DataLoader(SkeletonDS(Xeval, labels, idx, False), bs,
                                     shuffle=False, num_workers=4)
    c = t = 0
    for xb, yb in ld:
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            o = m(xb.to(DEVICE))
        c += (o.argmax(1).cpu() == yb).sum().item(); t += yb.numel()
    del m
    return 100.0 * c / t


_, te = SPLITS['reid_rand']
te = np.sort(te)

if 'their_eval' not in RB:
    reid_ckpt = RESULTS['B_reid_rand']['ckpt']       # trained on ORIGINAL skeletons
    RB['their_eval'] = {
        'reid_on_original': fixed_classifier_eval(reid_ckpt, X, te, SUBJ_LABEL),
        'reid_on_anonymised': fixed_classifier_eval(reid_ckpt, X_ANON, te, SUBJ_LABEL),
        'action_on_anonymised': fixed_classifier_eval(
            RESULTS['A_action_xsub']['ckpt'], X_ANON,
            np.sort(SPLITS['xsub'][1]), ACT_LABEL, num_classes=60)}
    save_rb()
t = RB['their_eval']
print('\n--- their protocol (fixed classifier, never retrained) ---')
print('  re-ID on original    %.2f%%' % t['reid_on_original'])
print('  re-ID on anonymised  %.2f%%   <- the headline-style figure' % t['reid_on_anonymised'])
print('  action on anonymised %.2f%%' % t['action_on_anonymised'])

In [ ]:
# ---- OUR evaluation: attackers retrained on the anonymised data ----
X_ORIG = X
try:
    X = X_ANON
    if 'our_eval_scratch' not in RB:
        RB['our_eval_scratch'] = run_experiment(
            tag='MOON_reid_scratch_anon', labels=SUBJ_LABEL, split='reid_rand',
            num_classes=40, epochs=80, lr=0.1, bs=64)
        save_rb()
    if 'our_eval_action' not in RB:
        RB['our_eval_action'] = run_experiment(
            tag='MOON_action_anon', labels=ACT_LABEL, split='xsub',
            num_classes=60, epochs=60, lr=0.1, bs=64)
        save_rb()

    if 'our_eval_probes' not in RB:
        @torch.no_grad()
        def feats(ckpt, idx, bs=256):
            m = BASE_STGCN(60).to(DEVICE)
            sd = _load_ckpt(ckpt)
            sd = sd.get('model', sd)
            m.load_state_dict({k: v for k, v in sd.items()
                               if not k.startswith('fc.')}, strict=False)
            m.eval()
            ld = torch.utils.data.DataLoader(SkeletonDS(X, SUBJ_LABEL, idx, False),
                                             bs, shuffle=False, num_workers=4)
            o = []
            for xb, _ in ld:
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    f_ = m.extract(xb.to(DEVICE))
                if f_.ndim > 2:
                    f_ = f_.mean(dim=tuple(range(2, f_.ndim)))
                o.append(f_.float().cpu().numpy())
            del m
            return np.concatenate(o, 0)

        tr_i, te_i = SPLITS['reid_rand']
        tr_i, te_i = np.sort(tr_i), np.sort(te_i)
        ck = RB['our_eval_action']['ckpt']
        Ftr, Fte = feats(ck, tr_i), feats(ck, te_i)
        ytr, yte = np.asarray(SUBJ_LABEL)[tr_i], np.asarray(SUBJ_LABEL)[te_i]
        sc = StandardScaler().fit(Ftr)
        Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)
        RB['our_eval_probes'] = {
            'linear': 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte),
            'mlp': 100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                       early_stopping=True, n_iter_no_change=15,
                                       random_state=1).fit(Ztr, ytr).score(Zte, yte)}
        save_rb()
finally:
    X = X_ORIG
    print('input array restored')

print('\n=== THE COMPARISON ===')
print('%-46s %8s' % ('evaluation of the same anonymised data', 're-ID'))
print('-' * 56)
print('%-46s %7.2f%%' % ('their protocol: fixed pre-trained classifier',
                         RB['their_eval']['reid_on_anonymised']))
print('%-46s %7.2f%%' % ('ours: linear probe, retrained encoder',
                         RB['our_eval_probes']['linear']))
print('%-46s %7.2f%%' % ('ours: MLP probe, retrained encoder',
                         RB['our_eval_probes']['mlp']))
print('%-46s %7.2f%%' % ('ours: from-scratch attacker on anonymised data',
                         RB['our_eval_scratch']['best_acc']))
print('-' * 56)
print('%-46s %7.2f%%' % ('reference: from-scratch on ORIGINAL data', 78.86))
print('\nMoon et al. report 4.20% for the fixed-classifier protocol.')

## 3 · M5 — NTU RGB+D 120

106 identities against 40, so chance falls from 2.50% to 0.94%. Our scaling law
(ratio to chance proportional to K^0.503, R-squared 0.98) predicts approximately
12.2% top-1 at 12.9 times chance for the linear probe. This measures it.

The download is 2.2 GB and the cache build takes roughly half an hour.

In [ ]:
P120 = os.path.join(DATA_DIR, 'ntu120_3danno.pkl')
URL120 = 'https://download.openmmlab.com/mmaction/pyskl/data/nturgbd/ntu120_3danno.pkl'
if not os.path.exists(P120):
    print('downloading NTU-120 annotations (2.2 GB) ...')
    rc = os.system('wget -c -q --show-progress -O "%s" "%s"' % (P120, URL120))
    assert rc == 0 and os.path.getsize(P120) > 1e9, 'download failed; retry the cell'
print('%.2f GB' % (os.path.getsize(P120) / 1e9))

import pickle
t0 = time.time()
with open(P120, 'rb') as fh:
    D120 = pickle.load(fh)
ann120 = D120['annotations'] if isinstance(D120, dict) else D120
print('loaded %d sequences in %.0f s' % (len(ann120), time.time() - t0))

names120 = [a['frame_dir'] for a in ann120]
setup120 = np.array([int(n[1:4]) for n in names120], dtype=np.int16)
subj120 = np.array([int(n[9:12]) for n in names120], dtype=np.int16)
act120 = np.array([int(n[17:20]) - 1 for n in names120], dtype=np.int16)
print('subjects %d  actions %d  setups %d'
      % (len(np.unique(subj120)), len(np.unique(act120)), len(np.unique(setup120))))

In [ ]:
CACHE120 = os.path.join(DATA_DIR, 'ntu120_T64_x.npy')
META120 = os.path.join(DATA_DIR, 'ntu120_T64_meta.npz')

if os.path.exists(CACHE120):
    print('cache exists')
else:
    buf = np.zeros((len(ann120), MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    keep = []
    for i, a in enumerate(ann120):
        kp = a.get('keypoint')
        if kp is None or kp.size == 0:
            continue
        try:
            buf[len(keep)] = resample(pre_normalize(kp.astype(np.float32)))
            keep.append(i)
        except Exception:
            continue
        if i % 5000 == 0:
            print('  %d / %d' % (i, len(ann120)), end='\r')
    buf = buf[:len(keep)]
    np.save(CACHE120, buf)
    np.savez(META120, setup=setup120[keep], subject=subj120[keep],
             action=act120[keep], names=np.array(names120)[keep])
    print('\ncached %d sequences' % len(keep))

X120 = np.load(CACHE120, mmap_mode='r')
M120 = np.load(META120, allow_pickle=True)
setup_1, subj_1, act_1 = M120['setup'], M120['subject'], M120['action']
SUBJ_IDS_120 = sorted(set(subj_1.tolist()))
S2L = {s: i for i, s in enumerate(SUBJ_IDS_120)}
SUBJ_LABEL_120 = np.array([S2L[s] for s in subj_1], dtype=np.int64)
ACT_LABEL_120 = act_1.astype(np.int64)
print('X120', X120.shape, '| %d identities, chance %.2f%%'
      % (len(SUBJ_IDS_120), 100.0 / len(SUBJ_IDS_120)))

XSUB120_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38,45,46,47,
                 49,50,52,53,54,55,56,57,58,59,70,74,78,80,81,82,83,84,85,86,89,
                 91,92,93,94,95,97,98,100,103]
rs = np.random.RandomState(SEED)
tr120, te120 = [], []
for s in SUBJ_IDS_120:
    idx = np.where(subj_1 == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr120 += idx[:k].tolist(); te120 += idx[k:].tolist()
SPLITS['reid_rand_120'] = (np.sort(np.array(tr120)), np.sort(np.array(te120)))
SPLITS['xsub_120'] = (np.where(np.isin(subj_1, XSUB120_TRAIN))[0],
                      np.where(~np.isin(subj_1, XSUB120_TRAIN))[0])
for k in ['reid_rand_120', 'xsub_120']:
    print('%-16s train %6d  test %6d' % (k, len(SPLITS[k][0]), len(SPLITS[k][1])))

In [ ]:
# core NTU-120 experiments: action model, from-scratch re-ID, ladder probes
X_ORIG, SL_ORIG, AL_ORIG = X, SUBJ_LABEL, ACT_LABEL
try:
    X, SUBJ_LABEL, ACT_LABEL = X120, SUBJ_LABEL_120, ACT_LABEL_120

    if 'A120_action' not in RB:
        RB['A120_action'] = run_experiment(tag='A120_action_xsub', labels=ACT_LABEL,
                                           split='xsub_120', num_classes=120,
                                           epochs=60, lr=0.1, bs=64)
        save_rb()
    if 'B120_reid' not in RB:
        RB['B120_reid'] = run_experiment(tag='B120_reid_rand', labels=SUBJ_LABEL,
                                         split='reid_rand_120', num_classes=106,
                                         epochs=80, lr=0.1, bs=64)
        save_rb()
    if 'D120_ft' not in RB:
        RB['D120_ft'] = run_experiment(tag='D120_finetune', labels=SUBJ_LABEL,
                                       split='reid_rand_120', num_classes=106,
                                       epochs=80, lr=0.05, bs=64,
                                       init_from=RB['A120_action']['ckpt'], freeze=False)
        save_rb()

    if 'ladder_120' not in RB:
        @torch.no_grad()
        def f120(ckpt, idx, bs=256):
            m = BASE_STGCN(120).to(DEVICE)
            sd = _load_ckpt(ckpt); sd = sd.get('model', sd)
            m.load_state_dict({k: v for k, v in sd.items()
                               if not k.startswith('fc.')}, strict=False)
            m.eval()
            ld = torch.utils.data.DataLoader(SkeletonDS(X, SUBJ_LABEL, idx, False),
                                             bs, shuffle=False, num_workers=4)
            o = []
            for xb, _ in ld:
                with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                    ff = m.extract(xb.to(DEVICE))
                if ff.ndim > 2:
                    ff = ff.mean(dim=tuple(range(2, ff.ndim)))
                o.append(ff.float().cpu().numpy())
            del m
            return np.concatenate(o, 0)

        tr, te = SPLITS['reid_rand_120']
        Ftr, Fte = f120(RB['A120_action']['ckpt'], tr), f120(RB['A120_action']['ckpt'], te)
        ytr, yte = SUBJ_LABEL[tr], SUBJ_LABEL[te]
        sc = StandardScaler().fit(Ftr)
        Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)
        RB['ladder_120'] = {
            'linear': 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte),
            'mlp': 100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                       early_stopping=True, n_iter_no_change=15,
                                       random_state=1).fit(Ztr, ytr).score(Zte, yte)}
        save_rb()
finally:
    X, SUBJ_LABEL, ACT_LABEL = X_ORIG, SL_ORIG, AL_ORIG
    print('arrays restored')

ch = 100.0 / 106
print('\n--- NTU RGB+D 120, 106 identities, chance %.2f%% ---' % ch)
print('action (X-Sub, 120 classes)  %.2f%%' % RB['A120_action']['best_acc'])
print('R5 from scratch              %.2f%%  (%.1fx chance)'
      % (RB['B120_reid']['best_acc'], RB['B120_reid']['best_acc'] / ch))
print('R2 linear probe              %.2f%%  (%.1fx)  predicted ~12.2%%, ~12.9x'
      % (RB['ladder_120']['linear'], RB['ladder_120']['linear'] / ch))
print('R3 MLP probe                 %.2f%%  (%.1fx)'
      % (RB['ladder_120']['mlp'], RB['ladder_120']['mlp'] / ch))
print('R4 fine-tune                 %.2f%%  (%.1fx)'
      % (RB['D120_ft']['best_acc'], RB['D120_ft']['best_acc'] / ch))

## 4 · Assemble tables

In [ ]:
L = []
if 'their_eval' in RB and 'our_eval_probes' in RB:
    L += ['Table B1. The same anonymised data under two evaluation protocols',
          '(NTU-60, 40 identities, chance 2.50%; anonymiser reimplemented from',
          ' Moon et al. (AAAI 2023), residual variant, alpha=1, beta=10)',
          '',
          '%-50s %9s' % ('evaluation', 're-ID'),
          '-' * 61,
          '%-50s %8.2f%%' % ('their protocol: fixed classifier, never retrained',
                             RB['their_eval']['reid_on_anonymised']),
          '%-50s %8.2f%%' % ('ours: R2 linear probe, retrained encoder',
                             RB['our_eval_probes']['linear']),
          '%-50s %8.2f%%' % ('ours: R3 MLP probe, retrained encoder',
                             RB['our_eval_probes']['mlp']),
          '%-50s %8.2f%%' % ('ours: R5 from scratch on anonymised data',
                             RB['our_eval_scratch']['best_acc']),
          '',
          '%-50s %8.2f%%' % ('reference: from scratch on original data', 78.86),
          '%-50s %8.2f%%' % ('their published figure (Shift-GCN)', 4.20),
          '%-50s %8.2f%%' % ('action accuracy on anonymised data',
                             RB['their_eval']['action_on_anonymised'])]

if 'ladder_120' in RB:
    ch = 100.0 / 106
    L += ['', 'Table B2. NTU RGB+D 120 (106 identities, chance 0.94%)', '',
          '%-30s %9s %10s' % ('measurement', 'top-1', 'x chance'),
          '-' * 52,
          '%-30s %8.2f%% %9.1fx' % ('R5 from scratch',
                                    RB['B120_reid']['best_acc'],
                                    RB['B120_reid']['best_acc'] / ch),
          '%-30s %8.2f%% %9.1fx' % ('R2 linear probe',
                                    RB['ladder_120']['linear'],
                                    RB['ladder_120']['linear'] / ch),
          '%-30s %8.2f%% %9.1fx' % ('R3 MLP probe',
                                    RB['ladder_120']['mlp'],
                                    RB['ladder_120']['mlp'] / ch),
          '%-30s %8.2f%% %9.1fx' % ('R4 fine-tune',
                                    RB['D120_ft']['best_acc'],
                                    RB['D120_ft']['best_acc'] / ch),
          '',
          'scaling-law prediction for R2 at K=106: ~12.2% top-1, ~12.9x chance',
          'action recognition (120 classes): %.2f%%' % RB['A120_action']['best_acc']]

txt = '\n'.join(L)
print(txt)
open(os.path.join(RES_B, 'revisionB_tables.txt'), 'w').write(txt)
save_rb()
print('\nwritten to', RES_B)